# Artificial Intelligence Technology and Application

## Deep Learning Lab Guide - Student Version

Independent implementation prepared by **Sundetkhan Bekzat**.


# 1 TextCNN Sentiment Analysis

This notebook keeps the lab objective but uses compact local examples so it can run without external datasets.


## 1.1 Tokenization
Text is transformed into fixed-size vectors before applying convolution-style pooling.


In [1]:
import re
import numpy as np

sentences = [
    "excellent product and fast delivery",
    "smooth support and helpful seller",
    "clean design with stable battery",
    "poor package and broken cable",
    "late delivery and bad support",
    "terrible quality with weak battery",
    "great camera and useful manual",
    "awful screen and slow refund",
]
labels = np.array([1, 1, 1, 0, 0, 0, 1, 0])

def tokens(text):
    return re.findall(r"[a-z]+", text.lower())

vocab = sorted({token for sentence in sentences for token in tokens(sentence)})
print("vocab size:", len(vocab))


vocab size: 31


## 1.2 Convolution-Style Features
Instead of a large framework, each sentence is represented by n-gram activation counts.


In [2]:
def ngram_features(text, vocab, widths=(1, 2, 3)):
    words = tokens(text)
    features = []
    for width in widths:
        grams = [" ".join(words[i:i + width]) for i in range(max(0, len(words) - width + 1))]
        features.append(len(grams))
        features.append(sum("good" in gram or "great" in gram or "excellent" in gram for gram in grams))
        features.append(sum("bad" in gram or "poor" in gram or "terrible" in gram or "awful" in gram for gram in grams))
    bag = [words.count(word) for word in vocab]
    return np.array(features + bag, dtype=float)

feature_matrix = np.vstack([ngram_features(sentence, vocab) for sentence in sentences])
print(feature_matrix.shape)


(8, 40)


## 1.3 Sentiment Head
The classifier consumes the pooled features similarly to the dense layer of TextCNN.


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score

text_head = LogisticRegression(max_iter=500).fit(feature_matrix, labels)
scores = cross_val_score(LogisticRegression(max_iter=500), feature_matrix, labels, cv=LeaveOneOut())
print("leave-one-out accuracy:", round(scores.mean(), 3))
sample = ngram_features("excellent support but slow delivery", vocab).reshape(1, -1)
print("positive probability:", round(text_head.predict_proba(sample)[0, 1], 3))


leave-one-out accuracy: 1.0
positive probability: 0.847


## 1.4 Important Terms
Weights attached to vocabulary terms show what drives positive sentiment.


In [4]:
term_weights = text_head.coef_[0][-len(vocab):]
ranked = sorted(zip(vocab, term_weights), key=lambda pair: abs(pair[1]), reverse=True)[:8]
print([(term, round(weight, 3)) for term, weight in ranked])


[('clean', np.float64(0.183)), ('design', np.float64(0.183)), ('stable', np.float64(0.183)), ('quality', np.float64(-0.179)), ('terrible', np.float64(-0.179)), ('weak', np.float64(-0.179)), ('helpful', np.float64(0.175)), ('seller', np.float64(0.175))]
